## Modified Stochastic Subspace Descent for Nonlinear Least Squares

### Gauss-Newton

Oracle: ${r_i(x)}^m_{i=1} = {q^\top \nabla r_i(x)}^m_{i=1} = J(x)q$ where q is a directional derivative. 

We know $r(x_k + p) = r(x_k) + J(x_k)p$ is the residual function. The nonlinear least square problem is then $$ \min_x ||r(x_k) + J(x_k)p||^2_2 $$

Apply $p = Pn$ to the SSD algorithm, where P is a projection matrix of size $d \times l$ where $l < d$. After applying the substitution, we obtain the normal equation 
\begin{equation}
-(JP)^\top r = (JP)^\top (JP) n
\end{equation}
solving for $n^*$ we have 
$$ Pn = p = P(P^\top J^\top J P)^{-1} P^\top J^\top r $$
Let $P = Q \cdot S$, where the column space of P and Q is the same and $S^{-1}$ exists. Then 
$$ p = Q(Q^\top J^\top J Q)^{-1} Q^\top J^\top r $$
Let $\tilde{J} = J \cdot Q$, then 
$$ p = Q ( \tilde{J}^\top \tilde{J} ) ^{-1} \tilde{J}^\top r$$
and the algorithm costs $l$ for $l$ directional derivatives. 

For computing purposes, compute by finding
$$ p = \text{arg} \min_{n \in \mathbb{R}} ||\tilde{J}n - r_x||^2_2$$

In [3]:
import jax
import jax.numpy as jnp
import pycutest
import numpy as np

In [8]:
def gn_ssd(x, l, r, alpha=1, max_iter=200):
    '''
    input: x is the starting point; P is dxl; r the residual function
    Q has Haar distribution
    Assume alpha = 1
    iteration: x = P 
    '''

    rx = r(x)
    for n in range(max_iter):
        Z = np.random.default_rng().standard_normal((dim, l))
        Q, R = linalg.qr(Z, mode='economic') #economic mode for finding the QR so it's not a full square matrix

        #use jax.jvp for jacobian-vector multiplication. Need x and v in tuple. Returns r(x) and Jx
        #use jax.vmap to map it to every row
        J_tild = jax.vmap(lambda v: jax.jvp(r, (x,), (v,))[1])(Q.T).T
        p = jnp.linalg.lstsq(J_tild.T, rx)[0]
        x = x - Q @ p

### Levenberg-Marquardt
L-M is a combination between Gauss-Newton and gradient descent. The problem is (after evaluating Lagrange Multiplier):
$$ \min_p ||Jp + r||^2 + \frac{\lambda}{2}||x_0|| $$
The normal equation for L-M is 
$$ -(JP)^\top r = ((JP)^\top (JP)-\lambda I) n $$
Input: 
* initial $\lambda_0$
* initial $x_0$ size $n \times 1$
* r

Importantly, need to update $\lambda$ each step. I use $D = I$ because we don't compute the whole Jacobian. 

Figure out how to update $\lambda$

In [ ]:
def lm_ssd(x, l, r, lam, max_iter=200):

    rx = r(x)
    for n in range(max_iter):
        Z = np.random.default_rng().standard_normal((dim, l))
        Q, R = linalg.qr(Z, mode='economic') #economic mode for finding the QR so it's not a full square matrix

        #use jax.jvp for jacobian-vector multiplication. Need x and v in tuple. Returns r(x) and Jx
        #use jax.vmap to map it to every row
        J_tild = jax.vmap(lambda v: jax.jvp(r, (x,), (v,))[1])(Q.T).T
        p = jnp.linalg.lstsq(J_tild, rx)[0]
        x = x - Q @ p